## Prompt engineering (Advanced)

<div style="text-align: right"> <b>KMYU</b></div>
<div style="text-align: right"> Initial issue : 2026.03.20 </div>
<div style="text-align: right"> last update : 2026.03.25 </div>

개정 이력  
- `2026.03.20` : 노트북 최초 작성

In [15]:
import os, sys

from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage

load_dotenv()

True

In [16]:
print(os.getenv("ANTHROPIC_MODEL"))

claude-haiku-4-5


### 1. 에이전트에 시스템 프롬프트 추가하기

In [17]:
agent = create_agent(model="claude-haiku-4-5")
response = agent.invoke(
    {"messages": HumanMessage(content="달의 수도는 어디야?")}
)
print(response['messages'][1].content)

달에는 수도가 없습니다. 달은 자연 위성이며, 현재 어떤 국가나 정부도 소유하지 않고 있습니다.

달에는 사람이 거주하지 않으며, 1969년 아폴로 11호 우주선이 달에 착륙한 이후 과학자들이 방문했을 뿐입니다.

혹시 다른 행성이나 위성에 대해 궁금하신 점이 있으신가요?


In [18]:
system_prompt = "너는 과학 소설 작가야. 사용자가 요청하는 수도에 대해서 창작해줘."
scifi_agent = create_agent(
    model="claude-haiku-4-5",
    system_prompt=system_prompt
    )

response = scifi_agent.invoke(
    {"messages": HumanMessage(content="달의 수도는 어디야?")}
)
print(response['messages'][1].content)

# 루나 시티 (Luna City)

달의 수도는 **루나 시티**라고 불러요. 위치는 달의 앞면(지구를 향한 면)의 **티코 분화구** 인근 지하에 있습니다.

## 🌙 주요 특징

**건축 구조**
- 약 500m 깊이의 용암굴을 개조한 지하 도시
- 투명한 돔 구조로 지표면 일부를 덮음
- 외부 방사선으로부터 보호되는 천연 요새

**인구 및 기능**
- 약 50만 명 거주 (2247년 기준)
- 국제 달 연합의 행정 중심지
- 우주 여행의 허브 역할

**주요 시설**
- 중력 조절 플랫폼 (화성 이민자 적응 센터)
- 저온 저장 도서관 (인류 문명 보존)
- 태양 관측소 (우주 기상 모니터링)
- 다중 배양 농장 (달 자체 식량 생산)

독특한 점은 지표면의 변이 없어서 항상 같은 별자리가 보인다는 거죠. 밤하늘이 변하지 않는 신비한 도시입니다. 🚀


### 2. CoT 실습

CoT(Chain of Thought, 생각의 사슬)는 모델에게 *"정답을 바로 말하지 말고, 풀이 과정을 단계별로 적은 다음 답하라"* 고 시키는 프롬프트 기법입니다.

사람도 어려운 문제를 풀 때 머릿속으로 한 단계씩 따져보면 실수가 줄어드는 것처럼, 모델도 추론 과정을 글로 적게 하면 더 정확한 답을 내놓는 경우가 많습니다.

아래에서는 **같은 문제**를 세 가지 방식으로 풀어보고 결과를 비교합니다.
- **CoT 없이**: 정답만 바로 말하게 하기
- **Zero-shot CoT**: 단계별로 생각한 뒤 답하게 하기
- **Few-shot CoT**: 풀이 예시를 보여준 뒤 답하게 하기

In [19]:
# CoT 효과를 확인할 추론 문제 (비전공자도 쉽게 검증할 수 있는 예시)
question = """공책 한 권과 연필 한 자루를 합친 가격은 1,100원입니다.
공책은 연필보다 1,000원 더 비쌉니다.
그렇다면 연필 한 자루는 얼마일까요?"""

# 정답: 연필 50원, 공책 1,050원  (합 1,100원, 차이 1,000원)
# 직관적으로 "100원"이라고 답하기 쉽지만, 차근차근 계산하면 50원입니다.


def ask(system_prompt: str, question: str):
    """시스템 프롬프트와 질문을 받아 모델에게 묻고, 프롬프트와 답변을 함께 출력한다."""
    agent = create_agent(model="claude-haiku-4-5", system_prompt=system_prompt)
    response = agent.invoke({"messages": HumanMessage(content=question)})

    print("[시스템 프롬프트]")
    print(system_prompt)
    print("\n[질문]")
    print(question)
    print("\n[모델 답변]")
    print(response["messages"][-1].content)
    print("-" * 60)
    return response

#### 2-1. CoT 없이 — 정답만 바로 답하기

먼저 풀이 과정 없이 **정답(숫자)만** 말하도록 시켜봅니다.
이렇게 하면 모델이 직관에 의존해 틀린 답(예: 100원)을 내놓기 쉽습니다.

In [20]:
direct_prompt = "너는 문제를 푸는 도우미야. 풀이 과정은 절대 적지 말고, 정답인 숫자만 한 줄로 답해."

_ = ask(direct_prompt, question)

[시스템 프롬프트]
너는 문제를 푸는 도우미야. 풀이 과정은 절대 적지 말고, 정답인 숫자만 한 줄로 답해.

[질문]
공책 한 권과 연필 한 자루를 합친 가격은 1,100원입니다.
공책은 연필보다 1,000원 더 비쌉니다.
그렇다면 연필 한 자루는 얼마일까요?

[모델 답변]
50
------------------------------------------------------------


#### 2-2. Zero-shot CoT — 단계별로 생각한 뒤 답하기

이번에는 *"정답을 말하기 전에 풀이 과정을 단계별로 적으라"* 고 지시합니다.
모델이 출력하는 **추론 과정**을 직접 눈으로 확인할 수 있습니다.

In [21]:
cot_prompt = (
    "너는 문제를 푸는 도우미야. "
    "정답을 말하기 전에 풀이 과정을 1단계, 2단계, 3단계처럼 번호를 붙여 "
    "차근차근 적은 뒤, 마지막 줄에 '정답:'이라고 쓰고 답을 말해."
)

_ = ask(cot_prompt, question)

[시스템 프롬프트]
너는 문제를 푸는 도우미야. 정답을 말하기 전에 풀이 과정을 1단계, 2단계, 3단계처럼 번호를 붙여 차근차근 적은 뒤, 마지막 줄에 '정답:'이라고 쓰고 답을 말해.

[질문]
공책 한 권과 연필 한 자루를 합친 가격은 1,100원입니다.
공책은 연필보다 1,000원 더 비쌉니다.
그렇다면 연필 한 자루는 얼마일까요?

[모델 답변]
# 연필 가격 구하기

**1단계: 미지수 정하기**
- 연필의 가격을 x원이라고 하겠습니다.

**2단계: 공책의 가격을 x로 나타내기**
- 공책은 연필보다 1,000원 더 비싸므로
- 공책의 가격 = (x + 1,000)원

**3단계: 방정식 세우기**
- 공책 + 연필 = 1,100원
- (x + 1,000) + x = 1,100

**4단계: 방정식 풀기**
- 2x + 1,000 = 1,100
- 2x = 1,100 - 1,000
- 2x = 100
- x = 50

**5단계: 검산**
- 연필: 50원
- 공책: 50 + 1,000 = 1,050원
- 합계: 50 + 1,050 = 1,100원 ✓

**정답: 연필 한 자루는 50원입니다.**
------------------------------------------------------------


#### 2-3. Few-shot CoT — 풀이 예시를 함께 보여주기

모델에게 *"이렇게 풀어라"* 는 **예시(풀이 과정이 담긴 본보기)** 를 한 개 보여준 뒤 새 문제를 풀게 하는 방법입니다.
모델은 예시의 풀이 방식을 따라 하므로, 추론 과정이 더 일정하고 안정적으로 나옵니다.

In [22]:
fewshot_prompt = """너는 문제를 푸는 도우미야. 아래 예시처럼 풀이 과정을 적은 뒤 정답을 말해.

[예시]
질문: 사과 한 개와 귤 한 개를 합치면 700원입니다. 사과는 귤보다 500원 더 비쌉니다. 귤은 얼마일까요?
풀이:
1단계: 귤 가격을 x라고 하면, 사과는 (x + 500)원이다.
2단계: 두 개를 합치면 x + (x + 500) = 700 이다.
3단계: 2x + 500 = 700 → 2x = 200 → x = 100 이다.
정답: 귤은 100원입니다.
"""

_ = ask(fewshot_prompt, question)

[시스템 프롬프트]
너는 문제를 푸는 도우미야. 아래 예시처럼 풀이 과정을 적은 뒤 정답을 말해.

[예시]
질문: 사과 한 개와 귤 한 개를 합치면 700원입니다. 사과는 귤보다 500원 더 비쌉니다. 귤은 얼마일까요?
풀이:
1단계: 귤 가격을 x라고 하면, 사과는 (x + 500)원이다.
2단계: 두 개를 합치면 x + (x + 500) = 700 이다.
3단계: 2x + 500 = 700 → 2x = 200 → x = 100 이다.
정답: 귤은 100원입니다.


[질문]
공책 한 권과 연필 한 자루를 합친 가격은 1,100원입니다.
공책은 연필보다 1,000원 더 비쌉니다.
그렇다면 연필 한 자루는 얼마일까요?

[모델 답변]
풀이:
1단계: 연필의 가격을 x원이라고 하면, 공책은 (x + 1,000)원이다.

2단계: 공책과 연필을 합치면 x + (x + 1,000) = 1,100 이다.

3단계: 2x + 1,000 = 1,100
      → 2x = 100
      → x = 50

정답: 연필 한 자루는 50원입니다.
------------------------------------------------------------


#### 2-4. 조금 더 복잡한 문제에 적용하기

이번에는 **직관적으로 답하기 쉬운 함정**이 있는 문제입니다.
많은 사람이 두 사람의 시간을 평균 내어 *"4시간 30분"* 이라고 답하지만, 실제 정답은 다릅니다.

앞에서 만든 `ask()` 함수와 프롬프트(`direct_prompt`, `cot_prompt`)를 **그대로 재사용**해서
**CoT 없이**와 **CoT 적용**의 결과를 비교해 봅니다.

In [23]:
# 직관적으로 틀리기 쉬운 '함께 일하기' 문제
question2 = """어떤 일을 혼자 하면 영희는 6시간, 철수는 3시간이 걸립니다.
두 사람이 함께 하면 그 일을 끝내는 데 몇 시간이 걸릴까요?"""

# 정답: 영희는 1시간에 1/6, 철수는 1시간에 1/3 만큼 일한다.
#       함께 하면 1시간에 1/6 + 1/3 = 1/2 만큼 → 일 전체를 끝내는 데 2시간.
#       (두 시간을 평균 낸 "4시간 30분"은 흔한 오답입니다.)

# 먼저 CoT 없이 — 정답만 바로 답하게 하기
_ = ask(direct_prompt, question2)

[시스템 프롬프트]
너는 문제를 푸는 도우미야. 풀이 과정은 절대 적지 말고, 정답인 숫자만 한 줄로 답해.

[질문]
어떤 일을 혼자 하면 영희는 6시간, 철수는 3시간이 걸립니다.
두 사람이 함께 하면 그 일을 끝내는 데 몇 시간이 걸릴까요?

[모델 답변]
2시간
------------------------------------------------------------


In [24]:
# 이번에는 CoT 적용 — 단계별로 생각하게 하기
_ = ask(cot_prompt, question2)

[시스템 프롬프트]
너는 문제를 푸는 도우미야. 정답을 말하기 전에 풀이 과정을 1단계, 2단계, 3단계처럼 번호를 붙여 차근차근 적은 뒤, 마지막 줄에 '정답:'이라고 쓰고 답을 말해.

[질문]
어떤 일을 혼자 하면 영희는 6시간, 철수는 3시간이 걸립니다.
두 사람이 함께 하면 그 일을 끝내는 데 몇 시간이 걸릴까요?

[모델 답변]
# 두 사람이 함께 일할 때 걸리는 시간

## 1단계: 각자의 작업 속도 구하기
- 영희: 1시간에 전체 일의 1/6을 완료
- 철수: 1시간에 전체 일의 1/3을 완료

## 2단계: 두 사람이 함께 일할 때의 작업 속도 구하기
1시간에 완료하는 일의 양 = 1/6 + 1/3

분모를 6으로 통일하면:
= 1/6 + 2/6
= 3/6
= 1/2

두 사람이 함께하면 1시간에 전체 일의 1/2을 완료합니다.

## 3단계: 전체 일을 끝내는 데 걸리는 시간 구하기
전체 일을 완료하려면 1/2씩 2번 해야 하므로:

걸리는 시간 = 1 ÷ (1/2) = 2시간

**정답: 2시간**
------------------------------------------------------------


#### 정리

- **CoT 없이**: 모델이 직관적으로 답해 실수(예: 100원)가 나오기 쉽습니다.
- **Zero-shot CoT**: *"단계별로 생각해줘"* 라는 한 문장만 추가해도 추론 과정이 드러나고 정답률이 올라갑니다.
- **Few-shot CoT**: 풀이 **예시**를 함께 주면 추론 과정이 더 일관되게 나옵니다.

> 핵심: 어려운 추론·계산 문제일수록 **"바로 답하지 말고 단계별로 생각하게"** 만드는 것만으로 답의 품질이 크게 좋아집니다.

### 3. ReAct 실습

지금까지는 모델이 **머릿속 지식만으로** 답했습니다. 하지만 '오늘 날짜'나 '지금 날씨'처럼 모델이 **직접 알 수 없는 정보**는 답할 수 없습니다.

**ReAct(Reasoning + Acting)** 는 모델이 *생각(Reasoning)* 만 하는 게 아니라, 필요할 때 **도구(함수)를 직접 호출(Acting)** 해서 정보를 얻고, 그 결과를 보고 다시 생각하는 방식입니다.

> 생각 → 도구 호출 → 결과 확인 → 다시 생각 → … → 최종 답변

이렇게 도구를 사용할 수 있는 모델을 **에이전트(Agent)** 라고 부릅니다.
이번 실습에서는 '오늘 날짜'와 '오늘 날씨'를 알려주는 도구 두 개를 만들고, 에이전트가 이 도구들을 **스스로 골라 호출하며** 답을 만들어가는 과정을 눈으로 확인합니다.

#### 3-1. 도구(Tool) 만들기

`@tool` 데코레이터를 함수 위에 붙이면, 그 함수를 에이전트가 호출할 수 있는 **도구**로 만들 수 있습니다.
함수의 **독스트링(설명문)** 은 매우 중요합니다. 모델은 이 설명을 읽고 *"언제 이 도구를 써야 하는지"* 를 판단하기 때문입니다.

- `get_today()` : 오늘 날짜를 알려줍니다.
- `get_weather(city)` : 도시 이름을 받아 오늘 날씨를 알려줍니다. (실제 날씨 API 대신 **목업**으로 서울 = 맑음 고정)

In [25]:
from datetime import date

from langchain.tools import tool


@tool
def get_today() -> str:
    """오늘 날짜를 'YYYY년 MM월 DD일' 형식으로 알려준다."""
    return date.today().strftime("%Y년 %m월 %d일")


@tool
def get_weather(city: str) -> str:
    """도시 이름을 입력받아 오늘 그 도시의 날씨를 알려준다. (실습용 목업 데이터)"""
    weather_db = {
        "서울": "맑음 (최고 24도 / 최저 14도)",
    }
    return weather_db.get(city, f"'{city}'의 날씨 정보가 없습니다.")

#### 3-2. 시스템 프롬프트와 에이전트 만들기

에이전트에게 **역할과 행동 규칙**을 시스템 프롬프트로 알려주고, 앞에서 만든 도구들을 `tools=[...]` 로 연결합니다.
여기서는 "날짜·날씨는 추측하지 말고 반드시 도구로 확인하라"고 지시해, 에이전트가 도구를 적극적으로 쓰도록 유도합니다.

In [26]:
react_system_prompt = """너는 한국어로 답하는 친절한 여행 도우미야.

답을 하기 위해 다음 도구를 쓸 수 있어:
- get_today: 오늘 날짜를 알려준다.
- get_weather: 특정 도시의 오늘 날씨를 알려준다.

행동 규칙:
1. 먼저 어떤 정보가 필요한지 생각한다.
2. 날짜와 날씨는 추측하지 말고 반드시 도구를 호출해 확인한다.
3. 도구로 얻은 사실을 바탕으로 답을 정리한다.

답변 형식:
- 먼저 '오늘 날짜'와 '서울 날씨'를 알려준다.
- 그다음 날씨에 어울리는 '2시간 내외 관광지' 2~3곳을 추천하고, 고른 이유를 한 줄로 덧붙인다.
- 날씨가 맑으면 야외 명소 위주로 추천한다."""

travel_agent = create_agent(
    model="claude-haiku-4-5",
    tools=[get_today, get_weather],
    system_prompt=react_system_prompt,
)

#### 3-3. 추론·도구 호출 과정 출력하기

에이전트가 답을 내놓을 때까지 주고받은 **모든 메시지**를 순서대로 출력하는 함수를 만듭니다.
이렇게 하면 '모델의 생각 → 도구 호출 → 도구 결과 → 최종 답변' 으로 이어지는 ReAct 과정을 한눈에 볼 수 있습니다.

- `[모델의 생각 / 답변]` : 모델이 작성한 추론 또는 최종 답
- `[도구 호출]` : 모델이 **어떤 도구를 어떤 인자로** 호출했는지
- `[도구 결과]` : 도구가 돌려준 값

In [27]:
from langchain.messages import HumanMessage, AIMessage, ToolMessage


def _to_text(content) -> str:
    """메시지 content가 문자열이든 블록 리스트든 텍스트만 뽑아낸다."""
    if isinstance(content, str):
        return content
    chunks = []
    for block in content or []:
        if isinstance(block, str):
            chunks.append(block)
        elif isinstance(block, dict) and block.get("type") == "text":
            chunks.append(block.get("text", ""))
    return "".join(chunks)


def run_react(agent, question):
    """에이전트를 실행하고 추론·도구 호출·도구 결과·최종 답변을 순서대로 출력한다."""
    result = agent.invoke({"messages": [HumanMessage(content=question)]})

    for msg in result["messages"]:
        if isinstance(msg, HumanMessage):
            print("[사용자 질문]")
            print(msg.content)
        elif isinstance(msg, AIMessage):
            text = _to_text(msg.content)
            if text.strip():
                print("[모델의 생각 / 답변]")
                print(text)
            for call in msg.tool_calls:
                print(f"[도구 호출] {call['name']} (인자: {call['args']})")
        elif isinstance(msg, ToolMessage):
            print(f"[도구 결과] {msg.name} → {msg.content}")
        print("-" * 60)

    return result

#### 3-4. 실행해보기

이제 *"오늘 서울 날씨를 확인하고, 2시간 내외로 관광할 여행지를 추천해줘"* 라고 물어봅니다.
에이전트가 `get_today` 와 `get_weather` 를 **스스로 호출**한 뒤, 그 결과를 바탕으로 추천을 만들어내는 과정을 살펴보세요.

In [28]:
user_question = "오늘 서울 날씨를 확인해주고, 2시간 내외로 관광할 여행지를 추천해줘."

_ = run_react(travel_agent, user_question)

[사용자 질문]
오늘 서울 날씨를 확인해주고, 2시간 내외로 관광할 여행지를 추천해줘.
------------------------------------------------------------
[모델의 생각 / 답변]
오늘의 정보를 확인해드리겠습니다!
[도구 호출] get_today (인자: {})
[도구 호출] get_weather (인자: {'city': '서울'})
------------------------------------------------------------
[도구 결과] get_today → 2026년 06월 23일
------------------------------------------------------------
[도구 결과] get_weather → 맑음 (최고 24도 / 최저 14도)
------------------------------------------------------------
[모델의 생각 / 답변]
완벽합니다! 정보를 정리해드립니다:

**📅 오늘 날짜:** 2026년 06월 23일  
**🌤️ 서울 날씨:** 맑음 (최고 24도 / 최저 14도)

---

**추천 관광지 (2시간 내외):**

1. **🏯 경복궁**
   - 이유: 맑은 날씨에 한옥과 역사 유산을 사진으로 담기 최적의 조건입니다.

2. **🌳 서울숲**
   - 이유: 6월의 쾌청한 날씨에 자연을 만끽하기 좋고, 산책로가 잘 정비되어 있습니다.

3. **🏘️ 북촌 한옥마을**
   - 이유: 골목길 산책이 쾌적하고, 한옥과 카페를 둘러보며 감성적인 사진을 찍을 수 있습니다.

오늘은 정말 좋은 날씨네요! 야외에서 서울의 아름다운 명소들을 충분히 즐길 수 있는 날입니다. 편한 신발을 신고 나가시길 추천합니다! 😊
------------------------------------------------------------


#### 정리

- **ReAct** 는 모델의 *생각(Reasoning)* 과 도구 *실행(Acting)* 을 번갈아 수행해, 모델이 모르는 정보(날짜·날씨 등)까지 처리합니다.
- `@tool` 데코레이터와 **좋은 독스트링**이 핵심입니다. 모델은 설명을 읽고 *언제 어떤 도구를 쓸지* 스스로 판단합니다.
- `create_agent(model, tools=[...], system_prompt=...)` 한 줄로 도구를 쓰는 에이전트를 만들 수 있습니다.
- 주고받은 메시지를 펼쳐 보면 에이전트의 **'생각 → 행동 → 관찰 → 답변'** 과정을 그대로 확인할 수 있습니다.

> 핵심: 도구를 쥐여 주면, 모델은 모르는 것을 **스스로 찾아보고** 답을 만들어냅니다. 이것이 에이전트의 출발점입니다.